# ⛓️ Blockchain Implementation — Colab Edition
**Advanced Level — Project 13**

Skills: Data Structures • Cryptography • Consensus

A blockchain built from scratch to show how the core pieces actually work:
1. **Data structures** — blocks linked by hash into an immutable chain
2. **Cryptography** — SHA-256 hashing, plus ECDSA digital signatures for transactions
3. **Consensus** — Proof-of-Work mining and the longest-valid-chain rule across simulated nodes

This is an educational implementation — it teaches the concepts clearly, but it's not production-grade (no real P2P networking, no fork-choice edge cases beyond the basics, no economic security analysis). It's the same simplification real "build your own blockchain" tutorials use.

## 1. Install dependencies

In [1]:
!pip install -q cryptography

## 2. Imports

In [2]:
import hashlib
import json
import time
from dataclasses import dataclass, field, asdict
from typing import List, Optional

from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature

## 3. Wallets — key pairs & digital signatures
Each wallet is an ECDSA key pair (the same family of algorithm Bitcoin and Ethereum use). The private key signs transactions; the public key lets anyone verify that signature without ever seeing the private key.

In [3]:
class Wallet:
    def __init__(self):
        self.private_key = ec.generate_private_key(ec.SECP256K1())
        self.public_key = self.private_key.public_key()

    @property
    def address(self):
        """A short, unique identifier derived from the public key (like a wallet address)."""
        pub_bytes = self.public_key.public_bytes(
            encoding=serialization.Encoding.X962,
            format=serialization.PublicFormat.UncompressedPoint,
        )
        return hashlib.sha256(pub_bytes).hexdigest()[:40]

    def sign(self, message: str) -> str:
        signature = self.private_key.sign(message.encode(), ec.ECDSA(hashes.SHA256()))
        return signature.hex()

    def public_key_pem(self) -> str:
        return self.public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo,
        ).decode()


def verify_signature(public_key_pem: str, message: str, signature_hex: str) -> bool:
    try:
        public_key = serialization.load_pem_public_key(public_key_pem.encode())
        public_key.verify(bytes.fromhex(signature_hex), message.encode(), ec.ECDSA(hashes.SHA256()))
        return True
    except InvalidSignature:
        return False

## 4. Transactions
A transaction moves value from one wallet address to another. It's signed by the sender so anyone can verify it was actually authorized, without needing to trust a central authority.

In [4]:
@dataclass
class Transaction:
    sender: str          # wallet address, or 'SYSTEM' for mining rewards
    recipient: str
    amount: float
    timestamp: float = field(default_factory=time.time)
    signature: Optional[str] = None
    sender_public_key: Optional[str] = None

    def message(self) -> str:
        return f"{self.sender}->{self.recipient}:{self.amount}:{self.timestamp}"

    def sign_with(self, wallet: Wallet):
        assert wallet.address == self.sender, "Only the sender's own wallet can sign this transaction"
        self.signature = wallet.sign(self.message())
        self.sender_public_key = wallet.public_key_pem()

    def is_valid(self) -> bool:
        if self.sender == "SYSTEM":            # mining rewards need no signature
            return True
        if not self.signature or not self.sender_public_key:
            return False
        return verify_signature(self.sender_public_key, self.message(), self.signature)

    def to_dict(self):
        return asdict(self)

## 5. Blocks
Each block bundles a batch of transactions and links to the previous block via its hash — that link is what makes the chain tamper-evident: changing any past block changes its hash, which breaks every block after it.

In [5]:
class Block:
    def __init__(self, index, transactions, previous_hash, timestamp=None, nonce=0):
        self.index = index
        self.timestamp = timestamp or time.time()
        self.transactions = transactions  # list[Transaction]
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        block_contents = {
            'index': self.index,
            'timestamp': self.timestamp,
            'transactions': [t.to_dict() for t in self.transactions],
            'previous_hash': self.previous_hash,
            'nonce': self.nonce,
        }
        block_string = json.dumps(block_contents, sort_keys=True, default=str)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def __repr__(self):
        return (f"Block #{self.index} | hash={self.hash[:12]}... | "
                f"prev={self.previous_hash[:12]}... | txs={len(self.transactions)} | nonce={self.nonce}")

## 6. The chain + Proof-of-Work consensus
Mining means repeatedly changing the `nonce` until the block's hash starts with `difficulty` zeros. That's deliberately expensive to compute but trivial to verify — the core trick behind Proof-of-Work: it makes rewriting history costly, since every block after the tampered one would need to be re-mined too.

In [6]:
class Blockchain:
    def __init__(self, difficulty=4, mining_reward=10.0):
        self.difficulty = difficulty
        self.mining_reward = mining_reward
        self.chain: List[Block] = [self._create_genesis_block()]
        self.pending_transactions: List[Transaction] = []

    def _create_genesis_block(self) -> Block:
        return Block(index=0, transactions=[], previous_hash="0" * 64)

    @property
    def last_block(self) -> Block:
        return self.chain[-1]

    def add_transaction(self, transaction: Transaction):
        if not transaction.is_valid():
            raise ValueError("Cannot add invalid/unsigned transaction to the pool")
        self.pending_transactions.append(transaction)

    def mine_pending_transactions(self, miner_address: str) -> Block:
        reward_tx = Transaction(sender="SYSTEM", recipient=miner_address, amount=self.mining_reward)
        transactions = self.pending_transactions + [reward_tx]

        block = Block(index=self.last_block.index + 1,
                       transactions=transactions,
                       previous_hash=self.last_block.hash)
        self._proof_of_work(block)

        self.chain.append(block)
        self.pending_transactions = []
        return block

    def _proof_of_work(self, block: Block):
        target = "0" * self.difficulty
        start = time.time()
        while not block.hash.startswith(target):
            block.nonce += 1
            block.hash = block.compute_hash()
        elapsed = time.time() - start
        print(f"Mined block #{block.index} in {elapsed:.2f}s | nonce={block.nonce} | hash={block.hash[:16]}...")

    def get_balance(self, address: str) -> float:
        balance = 0.0
        for block in self.chain:
            for tx in block.transactions:
                if tx.sender == address:
                    balance -= tx.amount
                if tx.recipient == address:
                    balance += tx.amount
        return balance

    def is_chain_valid(self, chain: Optional[List[Block]] = None) -> bool:
        chain = chain or self.chain
        for i in range(1, len(chain)):
            current, previous = chain[i], chain[i - 1]

            if current.hash != current.compute_hash():
                print(f"Block #{current.index} hash doesn't match its contents — tampered.")
                return False
            if current.previous_hash != previous.hash:
                print(f"Block #{current.index} doesn't correctly link to block #{previous.index}.")
                return False
            if not current.hash.startswith("0" * self.difficulty):
                print(f"Block #{current.index} doesn't satisfy the Proof-of-Work difficulty.")
                return False
            for tx in current.transactions:
                if not tx.is_valid():
                    print(f"Block #{current.index} contains an invalid transaction.")
                    return False
        return True

## 7. Demo — wallets, transactions, mining

In [7]:
chain = Blockchain(difficulty=4, mining_reward=10.0)

alice = Wallet()
bob = Wallet()
miner = Wallet()

print("Alice address:", alice.address)
print("Bob address:  ", bob.address)
print("Miner address:", miner.address)

Alice address: ae8be40f3d56dbd75773c74c2f937e118094aece
Bob address:   834c82912a4076cc879d8e75d6628088c59a6b11
Miner address: ab0ca19a780e944870c02ee56ce0f5c4eacaaa40


In [8]:
# Mine an initial block so Alice actually has a balance to spend
chain.add_transaction(Transaction(sender="SYSTEM", recipient=alice.address, amount=50.0))
chain.mine_pending_transactions(miner_address=miner.address)

print("\nAlice's balance:", chain.get_balance(alice.address))

Mined block #1 in 1.17s | nonce=33933 | hash=0000121de96f554b...

Alice's balance: 50.0


In [9]:
# Alice pays Bob — the transaction must be signed with Alice's private key
tx1 = Transaction(sender=alice.address, recipient=bob.address, amount=15.0)
tx1.sign_with(alice)

print("Transaction valid?", tx1.is_valid())
chain.add_transaction(tx1)

chain.mine_pending_transactions(miner_address=miner.address)

print("\nBalances after block #2:")
print("  Alice:", chain.get_balance(alice.address))
print("  Bob:  ", chain.get_balance(bob.address))
print("  Miner:", chain.get_balance(miner.address))

Transaction valid? True
Mined block #2 in 3.08s | nonce=80125 | hash=0000e65bcd58cfcf...

Balances after block #2:
  Alice: 35.0
  Bob:   15.0
  Miner: 20.0


In [10]:
print("Chain so far:")
for block in chain.chain:
    print(" ", block)

print("\nIs the chain valid?", chain.is_chain_valid())

Chain so far:
  Block #0 | hash=e7d3a118a00b... | prev=000000000000... | txs=0 | nonce=0
  Block #1 | hash=0000121de96f... | prev=e7d3a118a00b... | txs=2 | nonce=33933
  Block #2 | hash=0000e65bcd58... | prev=0000121de96f... | txs=2 | nonce=80125

Is the chain valid? True


## 8. Tampering demo
Shows *why* the chain is tamper-evident: editing a past block's data breaks its own hash and the link to the next block, so validation catches it immediately.

In [11]:
print("Valid before tampering:", chain.is_chain_valid())

# Sneakily try to give Bob extra coins by editing history directly
chain.chain[2].transactions[0].amount = 9000.0

print("Valid after tampering: ", chain.is_chain_valid())

# Even re-computing the tampered block's own hash won't fully save it —
# it still breaks the Proof-of-Work requirement and the transaction signature check.
chain.chain[2].hash = chain.chain[2].compute_hash()
print("Valid after re-hashing:", chain.is_chain_valid())

Valid before tampering: True
Block #2 hash doesn't match its contents — tampered.
Valid after tampering:  False
Block #2 doesn't satisfy the Proof-of-Work difficulty.
Valid after re-hashing: False


## 9. Consensus — longest valid chain wins
In a real network, multiple nodes each hold their own copy of the chain and may temporarily disagree (e.g. two miners solve a block at nearly the same time). The standard resolution rule: adopt whichever valid chain is longest — it represents the most cumulative Proof-of-Work, so it's treated as the canonical history.

In [12]:
import copy

def resolve_conflicts(nodes: List[Blockchain]) -> Blockchain:
    """Given several nodes' chains, return the longest one that's still valid."""
    longest = nodes[0]
    for node in nodes[1:]:
        if node.is_chain_valid() and len(node.chain) > len(longest.chain):
            longest = node
    return longest

# Simulate two nodes that briefly forked
node_a = copy.deepcopy(chain)
node_b = copy.deepcopy(chain)

# Node B mines one more block that Node A hasn't seen yet
node_b.add_transaction(Transaction(sender="SYSTEM", recipient=bob.address, amount=5.0))
node_b.mine_pending_transactions(miner_address=miner.address)

print(f"Node A length: {len(node_a.chain)} | Node B length: {len(node_b.chain)}")

winner = resolve_conflicts([node_a, node_b])
print(f"Network adopts the chain with length {len(winner.chain)} (Proof-of-Work / longest-chain rule)")

Mined block #3 in 0.27s | nonce=3580 | hash=00004d573c0d610d...
Node A length: 3 | Node B length: 4
Block #2 doesn't satisfy the Proof-of-Work difficulty.
Network adopts the chain with length 3 (Proof-of-Work / longest-chain rule)


## 10. Try adjusting mining difficulty
Higher difficulty = more leading zeros required = exponentially more hash attempts needed. This is the same lever real Proof-of-Work networks use to keep block times roughly constant as total mining power changes.

In [13]:
for difficulty in [2, 3, 4, 5]:
    test_chain = Blockchain(difficulty=difficulty)
    test_chain.add_transaction(Transaction(sender="SYSTEM", recipient=alice.address, amount=1.0))
    start = time.time()
    test_chain.mine_pending_transactions(miner_address=miner.address)
    print(f"difficulty={difficulty} -> {time.time() - start:.3f}s\n")

Mined block #1 in 0.02s | nonce=318 | hash=00e8fbb92c1c852c...
difficulty=2 -> 0.019s

Mined block #1 in 1.32s | nonce=8411 | hash=0000f6fb7d5fa982...
difficulty=3 -> 1.320s

Mined block #1 in 5.05s | nonce=99643 | hash=0000bad61c376e0c...
difficulty=4 -> 5.052s

Mined block #1 in 3.78s | nonce=111084 | hash=000007d8249c8b90...
difficulty=5 -> 3.782s



## Notes
- **This is a teaching model, not production infrastructure.** Real blockchains add peer-to-peer networking, Merkle trees for efficient transaction verification, UTXO or account-based state models, mempool fee markets, and much more robust fork-choice and network-partition handling.
- **ECDSA (SECP256K1)** is the same curve Bitcoin uses for signatures — chosen here so the cryptography maps directly onto real systems, not just a toy example.
- **Difficulty is fixed** in this notebook; real networks (like Bitcoin) auto-adjust it periodically to target a constant average block time as network hash power changes.
- **Proof-of-Work vs. Proof-of-Stake**: this notebook implements PoW because it's the more intuitive one to demonstrate with a mining loop. PoS (used by modern Ethereum) achieves consensus differently — via validators staking capital instead of burning compute — and would need a different implementation.